# Tech Challenge Fase 3 — Pipeline State of Data Brasil

Notebook consolidado com o pipeline completo de dados: ingestão (Bronze), harmonização de schema e tipagem (Silver), e construção das tabelas analíticas (Gold).

Este notebook contém o código final validado de cada etapa. Para o histórico de descobertas, bugs encontrados e decisões de design, ver `README.md`, que documenta o raciocínio por trás de cada escolha feita aqui.

**Ordem de execução:** as seções abaixo devem ser executadas em sequência.

## 1. Setup

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

import re
import unicodedata
from collections import Counter
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import (
    when, col, lit, coalesce, countDistinct, count,
    monotonically_increasing_id, sum as spark_sum
)
from pyspark.sql import Row

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

BUCKET = "gabrielsbn-techchallenge-fase3"


## 2. Camada Bronze — Parsing dos cabeçalhos

Os 3 anos da pesquisa usam dois formatos de cabeçalho diferentes:
- **2023-2024**: tupla serializada — `('P1_a ', 'Idade')`
- **2024-2025 / 2025-2026**: código e descrição concatenados — `1.a_idade`

O parser abaixo reconhece os dois formatos, com fallback para correções manuais pontuais (sujeira de fonte, ex.: parênteses desbalanceados).

In [ ]:
CORRECOES_MANUAIS = {
    "('P6_b_16 ', 'SQL Server Integration Services (SSIS))": ("P6_b_16", "SQL Server Integration Services (SSIS)")
}

PADRAO_FORMATO_A = re.compile(r"^\('([^']*)',\s*'(.*)'\)$")
PADRAO_FORMATO_B = re.compile(r"^(\d+(?:\.[a-zA-Z0-9]+)*)[\s_](.*)$")

def parse_coluna(nome_bruto):
    if nome_bruto in CORRECOES_MANUAIS:
        return CORRECOES_MANUAIS[nome_bruto]
    m = PADRAO_FORMATO_A.match(nome_bruto)
    if m:
        c, d = m.groups()
        return c.strip(), d.strip()
    m = PADRAO_FORMATO_B.match(nome_bruto)
    if m:
        c, d = m.groups()
        return c.strip(), d.strip()
    return None, nome_bruto

def carregar_e_parsear_colunas(caminho_s3):
    df = spark.read.csv(caminho_s3, header=True, sep=",", quote='"', escape='"', multiLine=True)
    return [parse_coluna(c) for c in df.columns]

colunas_2023_2024 = carregar_e_parsear_colunas(f"s3://{BUCKET}/bronze/state_of_data/ano=2023_2024/")
colunas_2024_2025 = carregar_e_parsear_colunas(f"s3://{BUCKET}/bronze/state_of_data/ano=2024_2025/")
colunas_2025_2026 = carregar_e_parsear_colunas(f"s3://{BUCKET}/bronze/state_of_data/ano=2025_2026/")

for nome, colunas in [("2023-2024", colunas_2023_2024), ("2024-2025", colunas_2024_2025), ("2025-2026", colunas_2025_2026)]:
    falhas = [c for c in colunas if c[0] is None]
    print(f"{nome}: {len(colunas)} colunas, {len(falhas)} falharam o parse")


## 3. Harmonização de schema (De-Para) — Bronze → Silver

O código de cada pergunta não é estável entre edições: quando uma pergunta é removida do meio de um bloco, todas as seguintes são reindexadas. Mapeamos essas cadeias manualmente, com base em investigação de schema drift (ver README seção 6-8).

Inclui também a correção específica da lista de Ferramentas de BI em 2023-2024, cuja numeração interna diverge da referência (2024-2025/2025-2026) por conta de uma remoção de itens que não aconteceu no mesmo ponto nas duas listas (README seção 20/BI).

In [ ]:
def normalizar_codigo(codigo):
    return codigo if codigo.startswith("P") else "P" + codigo.replace(".", "_")

CORRECOES_ANOS_ANTIGOS = {
    "P3_g": "P3_h",
    "P2_q": "P2_p", "P2_r": "P2_q", "P2_s": "P2_r", "P2_t": "P2_s",
}

DESLOCAMENTO_P4 = {'f': 'c', 'g': 'd', 'h': 'e', 'i': 'f', 'j': 'g', 'k': 'h', 'l': 'i', 'm': 'j'}
for la, ln in DESLOCAMENTO_P4.items():
    CORRECOES_ANOS_ANTIGOS[f"P4_{la}"] = f"P4_{ln}"
    for i in range(1, 40):
        CORRECOES_ANOS_ANTIGOS[f"P4_{la}_{i}"] = f"P4_{ln}_{i}"

CODIGOS_DESCONTINUADOS_P4 = ["P4_c", "P4_d", "P4_e"]
DESCONTINUADOS_COMPLETO = []
for base in CODIGOS_DESCONTINUADOS_P4:
    DESCONTINUADOS_COMPLETO.append(base)
    DESCONTINUADOS_COMPLETO += [f"{base}_{i}" for i in range(1, 40)]

ANOS_ANTIGOS = {"2023_2024", "2024_2025"}


def normalizar_texto(t):
    t = t.lower()
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()


descricoes_referencia_bi = {}
for codigo, desc in colunas_2024_2025:
    if codigo and codigo.startswith("4.j."):
        cod_norm = normalizar_codigo(codigo)
        cod_canonico = CORRECOES_ANOS_ANTIGOS.get(cod_norm, cod_norm)
        descricoes_referencia_bi[normalizar_texto(desc)] = cod_canonico

CORRECAO_BI_2023_2024 = {}
for codigo, desc in colunas_2023_2024:
    if codigo and codigo.startswith("P4_j_"):
        chave = normalizar_texto(desc)
        if chave in descricoes_referencia_bi:
            CORRECAO_BI_2023_2024[codigo] = descricoes_referencia_bi[chave]
        else:
            CORRECAO_BI_2023_2024[codigo] = f"{codigo}_descontinuada_bi"


def resolver_canonico(codigo_raw, ano_label):
    """Resolve o código canônico (2025-2026 como referência) para qualquer código bruto de qualquer ano."""
    cod_norm = normalizar_codigo(codigo_raw)
    if ano_label not in ANOS_ANTIGOS:
        return cod_norm  # 2025-2026 é a referência, nunca traduzir
    if ano_label == "2023_2024" and codigo_raw in CORRECAO_BI_2023_2024:
        return CORRECAO_BI_2023_2024[codigo_raw]
    if cod_norm in DESCONTINUADOS_COMPLETO:
        return f"{cod_norm}_descontinuada"
    return CORRECOES_ANOS_ANTIGOS.get(cod_norm, cod_norm)


## 4. Geração dos nomes finais de coluna

Nomes de coluna na Silver usam a descrição da pergunta (legível), não o código técnico — decisão tomada após confirmar que código não é uma chave estável entre edições (README seção 7). Colisões de nome (duas perguntas diferentes com a mesma descrição) são resolvidas checando se os códigos coexistem no mesmo ano: se nunca coexistem, são a mesma pergunta reindexada e são mescladas; se coexistem, são perguntas distintas e o nome é desambiguado com o código.

In [ ]:
def slugify(texto, max_len=60):
    texto = texto.strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return re.sub(r"_+", "_", texto).strip("_")[:max_len]

descricao_por_codigo_canonico = {}
for ano_colunas, ano_label in [(colunas_2025_2026, "2025_2026"), (colunas_2024_2025, "2024_2025"), (colunas_2023_2024, "2023_2024")]:
    for codigo, descricao in ano_colunas:
        if codigo is None:
            continue
        cod_canonico = resolver_canonico(codigo, ano_label)
        if cod_canonico not in descricao_por_codigo_canonico:
            descricao_por_codigo_canonico[cod_canonico] = descricao

slugs_brutos = {cod: slugify(desc) for cod, desc in descricao_por_codigo_canonico.items()}
contagem_slugs = Counter(slugs_brutos.values())
nome_final_por_codigo = {
    cod: (f"{slug}_{cod.lower()}" if contagem_slugs[slug] > 1 else slug)
    for cod, slug in slugs_brutos.items()
}
print(f"Total de nomes finais gerados: {len(nome_final_por_codigo)}")

def validar_nomes_unicos_por_ano(colunas_parseadas, ano_label):
    nomes = []
    for codigo, descricao in colunas_parseadas:
        if codigo is None:
            continue
        cod_canonico = resolver_canonico(codigo, ano_label)
        nomes.append(nome_final_por_codigo.get(cod_canonico, slugify(descricao)))
    dup = {n: c for n, c in Counter(nomes).items() if c > 1}
    print(f"{ano_label}: {len(dup)} duplicados" + (f" -> {dup}" if dup else ""))
    return len(dup) == 0

for cols, label in [(colunas_2023_2024, "2023_2024"), (colunas_2024_2025, "2024_2025"), (colunas_2025_2026, "2025_2026")]:
    validar_nomes_unicos_por_ano(cols, label)


## 5. Construção e persistência da camada Silver

`toDF(*novos_nomes)` renomeia colunas por posição — mais robusto que `.select().alias()`, que falha quando o nome antigo contém caracteres especiais (ponto, vírgula) que o Spark interpreta como expressão.

In [ ]:
def carregar_silver(caminho_s3, colunas_parseadas, ano_label):
    df = spark.read.csv(caminho_s3, header=True, sep=",", quote='"', escape='"', multiLine=True)
    novos_nomes = []
    for i, (codigo, descricao) in enumerate(colunas_parseadas):
        if codigo is None:
            novos_nomes.append(f"coluna_desconhecida_{i}")
            continue
        cod_canonico = resolver_canonico(codigo, ano_label)
        novos_nomes.append(nome_final_por_codigo.get(cod_canonico, slugify(descricao)))
    df_renomeado = df.toDF(*novos_nomes)
    return df_renomeado.withColumn("ano_pesquisa", lit(ano_label))

df_silver_2023_2024 = carregar_silver(f"s3://{BUCKET}/bronze/state_of_data/ano=2023_2024/", colunas_2023_2024, "2023_2024")
df_silver_2024_2025 = carregar_silver(f"s3://{BUCKET}/bronze/state_of_data/ano=2024_2025/", colunas_2024_2025, "2024_2025")
df_silver_2025_2026 = carregar_silver(f"s3://{BUCKET}/bronze/state_of_data/ano=2025_2026/", colunas_2025_2026, "2025_2026")

df_silver = (
    df_silver_2023_2024
    .unionByName(df_silver_2024_2025, allowMissingColumns=True)
    .unionByName(df_silver_2025_2026, allowMissingColumns=True)
)

CORRECOES_VALORES = {
    "faixa_salarial": {
        "de R$ 101/mês a R$ 2.000/mês": "de R$ 1.001/mês a R$ 2.000/mês",
        "de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30.000/mês",
    },
    "numero_de_funcionarios": {"de 501 a 100": "de 501 a 1.000"},
}
for coluna, mapa in CORRECOES_VALORES.items():
    df_silver = df_silver.replace(mapa, subset=[coluna])

colunas_analise = [c for c in df_silver.columns if c != "ano_pesquisa"]

def classificar_coluna(nome_coluna, df):
    valores = set(r[0] for r in df.select(nome_coluna).distinct().collect()) - {None}
    if valores <= {"0", "1"}:
        return "binaria"
    if valores and all(v.strip().lstrip("-").isdigit() for v in valores):
        return "numerica_candidata"
    return "categorica_texto"

resultado_classificacao = {c: classificar_coluna(c, df_silver) for c in colunas_analise}
colunas_binarias = [c for c, cat in resultado_classificacao.items() if cat == "binaria"]

for c in colunas_binarias:
    df_silver = df_silver.withColumn(c, when(col(c) == "1", True).when(col(c) == "0", False).otherwise(None))

df_silver = df_silver.withColumn("idade", col("idade").cast("int"))

df_silver.cache()
print(f"Total de linhas: {df_silver.count()}  |  Total de colunas: {len(df_silver.columns)}")

df_silver.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/silver/state_of_data_v4/"
)


## 6. Dicionário de dados

Identifica colunas "pai" de perguntas de múltipla escolha (texto concatenado, redundante com as colunas binárias filhas — descartadas da análise), separa categóricas nominais de ordinais, e persiste tudo como artefato reutilizável pela camada Gold.

In [ ]:
def codigo_pai_de(codigo):
    sufixo = "_descontinuada"
    tem_sufixo = codigo.endswith(sufixo)
    base = codigo[: -len(sufixo)] if tem_sufixo else codigo
    partes = base.rsplit("_", 1)
    if len(partes) == 2 and partes[1].isdigit():
        pai = partes[0]
        if tem_sufixo:
            pai += sufixo
        return pai
    return None

nome_para_codigo = {v: k for k, v in nome_final_por_codigo.items()}

codigos_pai_detectados = set()
for nome_col in colunas_binarias:
    codigo = nome_para_codigo.get(nome_col)
    if codigo:
        pai = codigo_pai_de(codigo)
        if pai:
            codigos_pai_detectados.add(pai)

categoricas_texto = [c for c, cat in resultado_classificacao.items() if cat == "categorica_texto"]
colunas_pai_final = [c for c in categoricas_texto if nome_para_codigo.get(c) in codigos_pai_detectados]
colunas_pai_final += [
    "quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalh",
    "quais_dessas_tecnologias_fazem_parte_do_seu_dia_a_dia_como_c",
]
colunas_categoricas_final = [c for c in categoricas_texto if c not in colunas_pai_final]

ORDEM_ORDINAL = {
    "faixa_idade": ["17-21", "22-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50-54", "55+"],
    "faixa_salarial": [
        "Menos de R$ 1.000/mês", "de R$ 1.001/mês a R$ 2.000/mês", "de R$ 2.001/mês a R$ 3.000/mês",
        "de R$ 3.001/mês a R$ 4.000/mês", "de R$ 4.001/mês a R$ 6.000/mês", "de R$ 6.001/mês a R$ 8.000/mês",
        "de R$ 8.001/mês a R$ 12.000/mês", "de R$ 12.001/mês a R$ 16.000/mês", "de R$ 16.001/mês a R$ 20.000/mês",
        "de R$ 20.001/mês a R$ 25.000/mês", "de R$ 25.001/mês a R$ 30.000/mês", "de R$ 30.001/mês a R$ 40.000/mês",
        "Acima de R$ 40.001/mês",
    ],
    "nivel": ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"],
    "nivel_de_ensino": [
        "Não tenho graduação formal", "Estudante de Graduação", "Graduação/Bacharelado",
        "Pós-graduação", "Mestrado", "Doutorado ou Phd", "Prefiro não informar",
    ],
    "numero_de_funcionarios": [
        "de 1 a 5", "de 6 a 10", "de 11 a 50", "de 51 a 100",
        "de 101 a 500", "de 501 a 1.000", "de 1.001 a 3.000", "Acima de 3.000",
    ],
    "numero_de_pessoas_em_dados": [
        "Ainda não temos pessoas atuando com dados na empresa",
        "1 - 3", "4 - 10", "11 - 20", "21 - 50", "51 - 100", "101 - 300", "Acima de 300 pessoas",
    ],
    "tempo_de_experiencia_em_dados": [
        "Não tenho experiência na área de dados", "Menos de 1 ano", "de 1 a 2 anos",
        "de 3 a 4 anos", "de 4 a 6 anos", "de 5 a 6 anos", "de 7 a 10 anos", "Mais de 10 anos",
    ],
    "tempo_de_experiencia_em_ti": [
        "Não tive experiência na área de TI/Engenharia de Software antes de começar a trabalhar na área de dados",
        "Menos de 1 ano", "de 1 a 2 anos", "de 3 a 4 anos", "de 5 a 6 anos", "de 7 a 10 anos", "Mais de 10 anos",
    ],
    "tempo_em_busca_de_oportunidade": ["0 - 6 meses", "7 meses - 1 ano", "1 ano - 2 anos", "acima de 2 anos"],
}

linhas_dicionario_rows = []
for nome_col in df_silver.columns:
    if nome_col == "ano_pesquisa":
        continue
    codigo_origem = nome_para_codigo.get(nome_col)
    descricao = descricao_por_codigo_canonico.get(codigo_origem)

    if nome_col == "idade":
        tipo_final, categoria = "int", "numerica"
    elif nome_col in colunas_binarias:
        tipo_final, categoria = "boolean", "binaria_multiescolha"
    elif nome_col in colunas_pai_final:
        tipo_final, categoria = "string", "pai_multiescolha_bruto"
    elif nome_col in ORDEM_ORDINAL:
        tipo_final, categoria = "string", "categorica_ordinal"
    else:
        tipo_final, categoria = "string", "categorica_nominal"

    if nome_col in ORDEM_ORDINAL:
        for pos, valor in enumerate(ORDEM_ORDINAL[nome_col]):
            linhas_dicionario_rows.append(Row(
                nome_coluna=nome_col, codigo_origem=codigo_origem, descricao=descricao,
                tipo_final=tipo_final, categoria=categoria, valor_categoria=valor, ordem_categoria=pos,
            ))
    else:
        linhas_dicionario_rows.append(Row(
            nome_coluna=nome_col, codigo_origem=codigo_origem, descricao=descricao,
            tipo_final=tipo_final, categoria=categoria, valor_categoria=None, ordem_categoria=None,
        ))

df_dicionario_dados = spark.createDataFrame(linhas_dicionario_rows)
df_dicionario_dados.write.mode("overwrite").parquet(f"s3://{BUCKET}/silver/_dicionario_dados/")
print(f"Dicionário de dados: {df_dicionario_dados.count()} linhas")


## 7. Camada Gold

Tabelas fato curadas (uma linha por respondente, sem pré-agregação) — cada uma cobre um tema de negócio específico, evitando redundância entre tabelas. Ver README seções 16-20 para o racional de cada decisão de escopo.

### 7.1 `gold_perfil_mercado`

In [ ]:
def adicionar_cargo_agrupado(df):
    return df.withColumn(
        "cargo_atual_agrupado",
        when(
            col("cargo_atual").isin(
                "Engenheiro de Dados/Data Engineer/Data Architect",
                "Arquiteto de Dados/Data Architect",
                "Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect",
            ),
            "Engenheiro de Dados/Arquiteto de Dados",
        ).otherwise(col("cargo_atual")),
    )

def adicionar_nivel_agrupado(df):
    return df.withColumn(
        "nivel_agrupado",
        when(col("nivel").isin("Especialista/Staff+", "Sênior"), "Sênior/Especialista").otherwise(col("nivel")),
    )

colunas_perfil_mercado = [
    "ano_pesquisa", "cargo_atual", "nivel", "atua_como_gestor", "situacao_de_trabalho",
    "setor", "numero_de_funcionarios", "modelo_de_trabalho_atual", "modelo_de_trabalho_ideal",
    "regiao_onde_mora", "uf_onde_mora", "genero", "cor_raca_etnia", "pcd",
    "faixa_idade", "nivel_de_ensino", "area_de_formacao",
    "tempo_de_experiencia_em_dados", "tempo_de_experiencia_em_ti",
]
df_gold_perfil_mercado = df_silver.select(*colunas_perfil_mercado)
df_gold_perfil_mercado = adicionar_cargo_agrupado(df_gold_perfil_mercado)
df_gold_perfil_mercado = adicionar_nivel_agrupado(df_gold_perfil_mercado)
df_gold_perfil_mercado.cache()
print(f"gold_perfil_mercado: {df_gold_perfil_mercado.count()} linhas")
df_gold_perfil_mercado.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/gold/perfil_mercado/"
)


### 7.2 `gold_remuneracao_por_perfil`

In [ ]:
VALOR_ESTIMADO_FAIXA_SALARIAL = {
    "Menos de R$ 1.000/mês": 750,
    "de R$ 1.001/mês a R$ 2.000/mês": 1500,
    "de R$ 2.001/mês a R$ 3.000/mês": 2500,
    "de R$ 3.001/mês a R$ 4.000/mês": 3500,
    "de R$ 4.001/mês a R$ 6.000/mês": 5000,
    "de R$ 6.001/mês a R$ 8.000/mês": 7000,
    "de R$ 8.001/mês a R$ 12.000/mês": 10000,
    "de R$ 12.001/mês a R$ 16.000/mês": 14000,
    "de R$ 16.001/mês a R$ 20.000/mês": 18000,
    "de R$ 20.001/mês a R$ 25.000/mês": 22500,
    "de R$ 25.001/mês a R$ 30.000/mês": 27500,
    "de R$ 30.001/mês a R$ 40.000/mês": 35000,
    "Acima de R$ 40.001/mês": 45000,
}

colunas_remuneracao = [
    "ano_pesquisa", "faixa_salarial", "cargo_atual", "nivel", "atua_como_gestor",
    "genero", "cor_raca_etnia", "regiao_onde_mora", "modelo_de_trabalho_atual",
    "tempo_de_experiencia_em_dados", "nivel_de_ensino",
]
df_gold_remuneracao = df_silver.select(*colunas_remuneracao)
df_gold_remuneracao = adicionar_cargo_agrupado(df_gold_remuneracao)
df_gold_remuneracao = adicionar_nivel_agrupado(df_gold_remuneracao)

expressao_valor = None
for faixa, valor in VALOR_ESTIMADO_FAIXA_SALARIAL.items():
    cond = when(col("faixa_salarial") == faixa, lit(valor))
    expressao_valor = cond if expressao_valor is None else expressao_valor.when(col("faixa_salarial") == faixa, lit(valor))
df_gold_remuneracao = df_gold_remuneracao.withColumn("salario_estimado", expressao_valor.otherwise(None))

df_gold_remuneracao.cache()
print(f"gold_remuneracao_por_perfil: {df_gold_remuneracao.count()} linhas")
df_gold_remuneracao.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/gold/remuneracao_por_perfil/"
)


### 7.3 `gold_diversidade`

In [ ]:
colunas_diversidade = [
    "ano_pesquisa", "genero", "cor_raca_etnia", "pcd", "cargo_atual", "nivel",
    "sim_devido_a_minha_cor_raca_etnia", "sim_devido_a_minha_identidade_de_genero", "sim_devido_ao_fato_de_ser_pcd",
]
df_gold_diversidade = df_silver.select(*colunas_diversidade)
df_gold_diversidade = adicionar_cargo_agrupado(df_gold_diversidade)
df_gold_diversidade.cache()
print(f"gold_diversidade: {df_gold_diversidade.count()} linhas")
df_gold_diversidade.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/gold/diversidade/"
)


### 7.4 `gold_adocao_tecnologias`

Formato longo (unpivot via `stack()`): cada grupo de colunas binárias vira duas colunas — `tecnologia` e `usa`.

In [ ]:
GRUPOS_TECNOLOGIA = {
    "Linguagem de Programação": "linguagem_de_programacao_dia_a_dia",
    "Banco de Dados": "banco_de_dados_dia_a_dia",
    "Cloud": "cloud_dia_a_dia",
    "Ferramenta de BI": "ferramenta_de_bi_dia_a_dia",
    "ETL (Engenheiro de Dados)": "ferramentas_etl_de",
    "ETL (Analista de Dados)": "ferramentas_etl_da",
    "Técnicas/Ferramentas de Ciência de Dados": "tecnologias_ds",
    "Ferramentas de Autonomia para Negócio": "ferramentas_autonomia_area_de_negocios",
}

df_dicionario_lookup = {r["nome_coluna"]: r["descricao"] for r in df_dicionario_dados.select("nome_coluna", "descricao").distinct().collect() if r["descricao"]}

df_gold_adocao_tecnologias = None
for categoria_label, nome_pai in GRUPOS_TECNOLOGIA.items():
    codigo_pai = nome_para_codigo.get(nome_pai)
    filhos = [c for c in colunas_binarias if codigo_pai_de(nome_para_codigo.get(c, "")) == codigo_pai]
    if not filhos:
        print(f"AVISO: nenhum filho para {categoria_label}")
        continue
    pares = [f"'{df_dicionario_lookup.get(c, c).replace(chr(39), '')}', `{c}`" for c in filhos]
    stack_expr = f"stack({len(filhos)}, {', '.join(pares)}) as (tecnologia, usa)"
    df_grupo = df_silver.selectExpr("ano_pesquisa", "cargo_atual as cargo_atual_agrupado", "nivel", stack_expr) \
        .withColumn("categoria", lit(categoria_label))
    df_gold_adocao_tecnologias = df_grupo if df_gold_adocao_tecnologias is None else df_gold_adocao_tecnologias.unionByName(df_grupo)

df_gold_adocao_tecnologias.cache()
print(f"gold_adocao_tecnologias: {df_gold_adocao_tecnologias.count()} linhas")
df_gold_adocao_tecnologias.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/gold/adocao_tecnologias/"
)


### 7.5 `gold_adocao_ia`

Inclui `cargo_unificado` (coalesce entre `cargo_atual` e `cargo_como_gestor`, já que a pergunta de prioridade de IA é respondida majoritariamente por gestores) e `respondente_id` (necessário para contar pessoas corretamente numa tabela em formato longo — `DISTINCT` nas colunas de contexto conta combinações, não pessoas; ver README 20.3).

In [ ]:
GRUPOS_ADOCAO_IA = {
    "Tipo de Uso (nível empresa)": "tipo_de_uso_de_ai_generativa_e_llm_na_empresa_p3_f",
    "Tipo de Uso (nível individual/produto)": "tipo_de_uso_de_ai_generativa_e_llm_na_empresa_p4_i",
    "Motivo de Não Adoção": "motivos_para_nao_usar_ai_generativa_e_llm",
    "Uso Pessoal (ChatGPT/Copilot)": "usa_chatgpt_ou_copilot_no_trabalho",
}

df_silver_com_id = df_silver.withColumn("respondente_id", monotonically_increasing_id())
df_silver_com_id = df_silver_com_id.withColumn(
    "atua_como_gestor",
    when(col("atua_como_gestor").isin("1", "TRUE", "true"), True)
    .when(col("atua_como_gestor").isin("0", "FALSE", "false"), False)
    .otherwise(None),
)

df_gold_adocao_ia = None
for categoria_label, nome_pai in GRUPOS_ADOCAO_IA.items():
    codigo_pai = nome_para_codigo.get(nome_pai)
    filhos = [c for c in colunas_binarias if codigo_pai_de(nome_para_codigo.get(c, "")) == codigo_pai]
    if not filhos:
        continue
    pares = [f"'{df_dicionario_lookup.get(c, c).replace(chr(39), '')}', `{c}`" for c in filhos]
    stack_expr = f"stack({len(filhos)}, {', '.join(pares)}) as (item, valor)"
    df_grupo = df_silver_com_id.selectExpr(
        "respondente_id", "ano_pesquisa",
        "cargo_atual as cargo_atual_agrupado", "cargo_como_gestor", "atua_como_gestor", "nivel",
        "ai_generativa_e_llm_e_uma_prioridade", "empresa_esta_conseguindo_ter_bons_resultados_com_llms",
        stack_expr,
    ).withColumn("categoria", lit(categoria_label))
    df_gold_adocao_ia = df_grupo if df_gold_adocao_ia is None else df_gold_adocao_ia.unionByName(df_grupo)

df_gold_adocao_ia = df_gold_adocao_ia.withColumn("cargo_unificado", coalesce(col("cargo_atual_agrupado"), col("cargo_como_gestor")))

df_gold_adocao_ia.cache()
print(f"gold_adocao_ia: {df_gold_adocao_ia.count()} linhas")
df_gold_adocao_ia.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(
    f"s3://{BUCKET}/gold/adocao_ia/"
)


## 8. Consultas Analíticas

As consultas abaixo reproduzem, em PySpark, a mesma lógica das queries Athena usadas para gerar os números e gráficos do material executivo — organizadas pela pergunta de negócio que respondem. Podem ser executadas diretamente sobre os DataFrames Gold já construídos na Seção 7, sem precisar do Athena.

### 8.1 Estrutura do mercado — cargos, senioridade e modelo de trabalho

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import sum as spark_sum, round as spark_round

def distribuicao_percentual(df, coluna_dimensao):
    contagem = df.filter(col(coluna_dimensao).isNotNull()).groupBy("ano_pesquisa", coluna_dimensao).count()
    total_por_ano = Window.partitionBy("ano_pesquisa")
    return contagem.withColumn(
        "percentual", spark_round(100.0 * col("count") / spark_sum("count").over(total_por_ano), 1)
    ).orderBy(col("ano_pesquisa").desc(), col("count").desc())

print("=== Distribuição de cargos (Slide 4) ===")
distribuicao_percentual(df_gold_perfil_mercado, "cargo_atual_agrupado").show(15, truncate=False)

print("=== Distribuição de senioridade (Slide 5) ===")
distribuicao_percentual(df_gold_perfil_mercado, "nivel_agrupado").show(10, truncate=False)

print("=== Modelo de trabalho atual (Slide 6) ===")
distribuicao_percentual(df_gold_perfil_mercado, "modelo_de_trabalho_atual").show(15, truncate=False)


### 8.2 Remuneração por cargo e senioridade (Slide 7)

In [ ]:
from pyspark.sql.functions import avg, count as spark_count

df_gold_remuneracao.filter(
    col("salario_estimado").isNotNull() & col("cargo_atual_agrupado").isNotNull() & col("nivel_agrupado").isNotNull()
).groupBy("ano_pesquisa", "cargo_atual_agrupado", "nivel_agrupado").agg(
    spark_round(avg("salario_estimado"), 0).alias("salario_medio_estimado"),
    spark_count("*").alias("total_respondentes"),
).filter(col("total_respondentes") >= 10).orderBy(
    col("ano_pesquisa").desc(), col("salario_medio_estimado").desc()
).show(20, truncate=False)


### 8.3 Diversidade — composição de gênero e percepção de prejuízo (Slides 9-10)

In [ ]:
print("=== Composição de gênero por ano ===")
distribuicao_percentual(df_gold_diversidade, "genero").show(15, truncate=False)

print("=== Percentual de prejuízo profissional percebido, por fator ===")
df_gold_diversidade.agg(
    spark_round(100.0 * spark_sum(when(col("sim_devido_a_minha_identidade_de_genero"), 1).otherwise(0)) / spark_count("*"), 1).alias("pct_prejuizo_genero"),
    spark_round(100.0 * spark_sum(when(col("sim_devido_a_minha_cor_raca_etnia"), 1).otherwise(0)) / spark_count("*"), 1).alias("pct_prejuizo_raca"),
    spark_round(100.0 * spark_sum(when(col("sim_devido_ao_fato_de_ser_pcd"), 1).otherwise(0)) / spark_count("*"), 1).alias("pct_prejuizo_pcd"),
).show()


### 8.4 Tecnologias em adoção — linguagens, cloud, bancos de dados e BI (Slides 11-13)

In [ ]:
def ranking_tecnologia(categoria, excluir_ano=None, limite=10):
    df_filtrado = df_gold_adocao_tecnologias.filter(col("categoria") == categoria)
    if excluir_ano:
        df_filtrado = df_filtrado.filter(col("ano_pesquisa") != excluir_ano)
    return df_filtrado.groupBy("ano_pesquisa", "tecnologia").agg(
        spark_sum(when(col("usa"), 1).otherwise(0)).alias("total_usa"),
        spark_count("*").alias("total_respondentes"),
        spark_round(100.0 * spark_sum(when(col("usa"), 1).otherwise(0)) / spark_count("*"), 1).alias("percentual_adocao"),
    ).orderBy(col("ano_pesquisa").desc(), col("percentual_adocao").desc()).limit(limite)

print("=== Linguagens de programação (descontinuada em 2025-2026) ===")
ranking_tecnologia("Linguagem de Programação", excluir_ano="2025_2026").show(truncate=False)

print("=== Cloud ===")
ranking_tecnologia("Cloud").show(truncate=False)

print("=== Banco de Dados ===")
ranking_tecnologia("Banco de Dados").show(truncate=False)

print("=== Ferramenta de BI ===")
ranking_tecnologia("Ferramenta de BI").show(truncate=False)


### 8.5 Adoção de Inteligência Artificial (Slides 14-15)

Usa `respondente_id` (não as colunas de contexto) para contar pessoas — numa tabela em formato longo, `DISTINCT` nas colunas de contexto conta combinações, não pessoas (ver Seção 7.5).

In [ ]:
print("=== Crescimento da priorização de IA entre gestores ===")
df_gold_adocao_ia.filter(col("ai_generativa_e_llm_e_uma_prioridade").isNotNull()).select(
    "respondente_id", "ano_pesquisa", "ai_generativa_e_llm_e_uma_prioridade"
).distinct().groupBy("ano_pesquisa").agg(
    spark_round(100.0 * spark_sum(when(col("ai_generativa_e_llm_e_uma_prioridade").startswith("Sim"), 1).otherwise(0)) / spark_count("*"), 1).alias("pct_ia_prioridade"),
    spark_count("*").alias("total_respondentes_unicos"),
).orderBy("ano_pesquisa").show()

print("=== Motivos de não adoção de IA ===")
df_gold_adocao_ia.filter(col("categoria") == "Motivo de Não Adoção").groupBy("item").agg(
    spark_sum(when(col("valor"), 1).otherwise(0)).alias("total_marcou"),
    spark_sum(when(col("valor").isNotNull(), 1).otherwise(0)).alias("total_respondentes_validos"),
).withColumn(
    "percentual", spark_round(100.0 * col("total_marcou") / col("total_respondentes_validos"), 1)
).orderBy(col("percentual").desc()).show(truncate=False)


### 8.6 Diferenças regionais e de modelo de trabalho (Slides 16-17)

In [ ]:
print("=== Concentração de profissionais por região ===")
df_gold_perfil_mercado.filter(col("regiao_onde_mora").isNotNull()).groupBy("regiao_onde_mora").agg(
    spark_count("*").alias("total_profissionais")
).withColumn(
    "percentual_concentracao", spark_round(100.0 * col("total_profissionais") / df_gold_perfil_mercado.count(), 1)
).orderBy(col("total_profissionais").desc()).show(truncate=False)

print("=== Salário médio estimado por região ===")
df_gold_remuneracao.filter(col("regiao_onde_mora").isNotNull() & col("salario_estimado").isNotNull()).groupBy(
    "regiao_onde_mora"
).agg(
    spark_count("*").alias("total_profissionais"),
    spark_round(avg("salario_estimado"), 0).alias("salario_medio_estimado"),
).orderBy(col("salario_medio_estimado").desc()).show(truncate=False)

print("=== Modelo de trabalho: atual x desejado ===")
df_gold_perfil_mercado.filter(
    col("modelo_de_trabalho_atual").isNotNull() & col("modelo_de_trabalho_ideal").isNotNull()
).groupBy("modelo_de_trabalho_atual", "modelo_de_trabalho_ideal").agg(
    spark_count("*").alias("total")
).orderBy(col("total").desc()).show(20, truncate=False)
